# Image Cluster Notebook
This notebook is designed to create labels for NAC images through K-Means clustering. 

## Imports

In [ ]:
# PROJ must be configured before rasterio/localtileserver are imported.
import os
os.environ["PROJ_IGNORE_CELESTIAL_BODY"] = "YES"

from pathlib import Path
import sys
import ipysheet
from IPython.display import Markdown, display
import ipywidgets
import leafmap
import numpy
import pandas
import rasterio
from rasterio.windows import Window
from localtileserver import TileClient, get_leaflet_tile_layer
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from ipyleaflet import WidgetControl

repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from model.clustering.Clusterer import Clusterer
from model.clustering.ImageHelperSingleBand import ImageHelper

# Configuration

`inFile`: Input single-band NAC file to perform clustering on. 

`noDataValue`: Nodata value to ignore in clustering. This is vital for the clustering algorithm to properly capture the valid data distribution. 

`numClusters`: Number of K-means clusters to create. Higher values result in a more noisy output (making it harder to discern between cluster groups by eye), while lower values create more homogenous clusters (which makes for poor labels). 

`cropSize`: Target crop size of input image; image is cropped to size in the center to improve performance of the algorithm and display tools. 

In [ ]:
# Original full-resolution lunar raster.
inFile = "/explore/nobackup/projects/lfm/Benchmarks/Craters/NAC_PHO_E064S3160/NAC_DTM_NEWCRATER6_M1219245090_80CM.TIF"
inFile = Path(inFile)

NAC_NODATA = -3.40282265508890445e+38
noDataValue = NAC_NODATA

# Number of K-Means clusters to use
numClusters = 20

# Crop size of input image, clustering will run on this smaller image for demonstration purposes
cropSize = 512

# Outputs are written here.
outDirectory = Path(".")
outDirectory.mkdir(parents=True, exist_ok=True)

clippedInputFile = outDirectory / (
    f"{inFile.stem}-clip-{cropSize}{inFile.suffix}"
)
labelsFile = outDirectory / (
    f"{inFile.stem}-clip-{cropSize}-labels{inFile.suffix}"
)
clusterMapFile = outDirectory / (
    f"{inFile.stem}-clip-{cropSize}-cluster-map{inFile.suffix}"
)

# Helper functions

In [ ]:
def crop_center(src_path: Path, dst_path: Path, size: int = 512) -> Path:
    """Write a centered square crop while preserving CRS/georeferencing."""
    with rasterio.open(src_path) as src:
        if src.width < size or src.height < size:
            raise ValueError(
                f"Raster is only {src.width}x{src.height}; "
                f"cannot make a {size}x{size} crop."
            )

        col_off = (src.width - size) // 2
        row_off = (src.height - size) // 2
        window = Window(col_off, row_off, size, size)

        data = src.read(window=window)
        transform = src.window_transform(window)

        profile = src.profile.copy()
        profile.update(
            width=size,
            height=size,
            transform=transform,
        )

        with rasterio.open(dst_path, "w", **profile) as dst:
            dst.write(data)

    return dst_path


# ----------------------------------------------------------------------------
# handleClick
# ----------------------------------------------------------------------------
def handleClick(change: dict) -> None:
    with output:
        if change.new == "Next":
            if not sl.value:
                print("Select at least one cluster before clicking Next.")
                bt.value = "Select:"
                return

            nn = updateList(list(sl.options), list(sl.value))
            updateDict("N")
            sl.options = nn
            bt.value = "Select:"

        if change.new == "Done":
            updateDict("D")

        if change.new == "Start Over":
            sl.options = opts
            updateDict("S")
            bt.value = "Select:"


# ----------------------------------------------------------------------------
# relabel
# ----------------------------------------------------------------------------
def relabel(labelArray: numpy.ndarray, lookup: dict) -> numpy.ndarray:
    newLab = labelArray.copy()

    for k, v in lookup.items():
        if len(v) == 1 and k == v[0]:
            continue
        newLab = numpy.where(numpy.isin(newLab, v), k, newLab)

    return newLab


# ----------------------------------------------------------------------------
# updateDict
# ----------------------------------------------------------------------------
def updateDict(op: str) -> None:
    if op == "N":
        key = list(sl.value)[0]
        table[key] = list(sl.value)

    if op == "D":
        if len(sl.options) > 0:
            key = list(sl.options)[0]
            table[key] = list(sl.options)

        print("Final Groups : ", table)

    if op == "S":
        table.clear()


# ----------------------------------------------------------------------------
# updateList
# ----------------------------------------------------------------------------
def updateList(old: list, out: list) -> list:
    return [ele for ele in old if ele not in out]


# Step 1: Clip the input raster

In [ ]:
# This is intentionally performed BEFORE ImageHelper ingestion or clustering.
# The full input TIFF is never passed to Clusterer.getClusters().
crop_center(
    src_path=inFile,
    dst_path=clippedInputFile,
    size=cropSize,
)

print(f"Full input:    {inFile}")
print(f"Clipped input: {clippedInputFile}")

# Step 2: Ingest ONLY the clipped raster

In [ ]:
inHelper = ImageHelper()
inHelper.initFromFile(
    inputFile=clippedInputFile,
    noDataValue=noDataValue,
)

print(f"Clustering input shape: {inHelper.getBand().shape}")

# Step 3: Generate first-pass clusters on the clipped raster

In [ ]:
# Add singleton dimension for the single input band: (H, W) -> (1, H, W).
# For cropSize=512, clustering operates on only 512x512 pixels.
labels = Clusterer.getClusters(
    bands=numpy.expand_dims(inHelper.getBand(), axis=0),
    numClusters=numClusters,
)

# Because inHelper was created from clippedInputFile, the label GeoTIFF is
# automatically written with the same clipped extent/transform/CRS.
labelsDs = Clusterer.labelsToGeotiff(
    inHelper._dataset,
    labelsFile,
    labels,
)

lHelper = ImageHelper()
lHelper.initFromDataset(labelsDs, noDataValue)

print(f"Clipped labels: {labelsFile}")

# Step 4: Display clipped image + clipped labels

In [ ]:
# Use localtileserver directly for raster serving. This path works with the
# Jupyter/VS Code loopback bridge and avoids leafmap.add_raster().
image_client = TileClient(str(clippedInputFile), debug=True)
labels_client = TileClient(str(labelsFile), debug=True)

image_layer = get_leaflet_tile_layer(
    image_client,
    vmin=inHelper._minValue,
    vmax=inHelper._maxValue,
    nodata=inHelper._noDataValue,
    opacity=1.0,
)
image_layer.name = clippedInputFile.name

labels_layer = get_leaflet_tile_layer(
    labels_client,
    vmin=lHelper._minValue,
    vmax=lHelper._maxValue,
    nodata=lHelper._noDataValue,
    opacity=0.5,
    colormap="viridis",
)
labels_layer.name = labelsFile.name

print("Image bounds:", image_layer.bounds)
print("Labels bounds:", labels_layer.bounds)

# TEMPORARY TEST: don't let Leaflet restrict tile loading by bounds
image_layer.bounds = None
labels_layer.bounds = None

m = leafmap.Map(
    fullscreen_control=False,
    layers_control=True,
    search_control=False,
    draw_control=False,
    measure_control=False,
    scale_control=False,
    toolbar_control=True,
    center=image_client.center(),
    zoom=image_client.default_zoom,
)

# Remove the default Earth/OpenStreetMap basemap.
m.remove(m.layers[0])

m.add(image_layer)
m.add(labels_layer)

m.layout.height = "600px"

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import ipywidgets as widgets
from ipyleaflet import WidgetControl

# Unique cluster IDs
cluster_ids = sorted(int(i) for i in numpy.unique(labels))
vmin = min(cluster_ids)
vmax = max(cluster_ids)

cmap = cm.get_cmap("viridis")

rows = []
for cid in cluster_ids:
    # Match the same continuous viridis mapping used by the layer
    if vmax == vmin:
        t = 0.5
    else:
        t = (cid - vmin) / (vmax - vmin)

    hex_color = mcolors.to_hex(cmap(t))

    rows.append(
        f"""
        <div style="display:flex; align-items:center; margin:2px 0;">
            <div style="
                width:18px;
                height:12px;
                background:{hex_color};
                border:1px solid #444;
                margin-right:8px;
                flex:0 0 auto;
            "></div>
            <div style="font-size:12px;">Cluster {cid}</div>
        </div>
        """
    )

legend_html = widgets.HTML(
    value=f"""
    <div style="
        background:white;
        color: #111;
        padding:8px 10px;
        border:1px solid #777;
        border-radius:4px;
        max-height:300px;
        min-width:140px;
        overflow-y:auto;
        box-shadow:0 1px 4px rgba(0,0,0,0.25);
    ">
        <div style="font-weight:bold; margin-bottom:6px;">
            Cluster legend
        </div>
        {''.join(rows)}
    </div>
    """
)

legend_control = WidgetControl(widget=legend_html, position="topright")
m.add(legend_control)
display(m)

## Update the labels
Select multiple values by clicking the mouse or using the arrow keys while pressing Shift, Control, or Command.

In [ ]:
opts = list(numpy.unique(labels))

sl = ipywidgets.SelectMultiple(
    options=opts,
    layout=ipywidgets.Layout(height="200px", width="150px"),
)

bt = ipywidgets.ToggleButtons(
    options=["Select:", "Next", "Done", "Start Over"],
    value="Select:",
)

output = ipywidgets.Output()
display(sl, bt, output)
table = {}
bt.observe(handleClick, names="value")

## Edit the groups
Edit cluster IDs in each group. When finished, proceed to the next cell.

In [ ]:
strTab = {}

for item in table:
    strTab[item] = ", ".join(str(i) for i in table[item])

df = pandas.DataFrame(strTab.items(), columns=["Class", "Cluster ID"])
sheet = ipysheet.from_dataframe(df)
sheet.column_width = [1, 5]
sheet

In [ ]:
editedDf = ipysheet.to_dataframe(sheet)
strClusters = editedDf.to_dict()["Cluster ID"]

finalClusters = {}

for key in strClusters:
    strCluster = strClusters[key]
    finalClusters[int(key)] = [
        int(i.strip()) for i in strCluster.split(",") if i.strip()
    ]

print(finalClusters)
newClusters = relabel(labels, finalClusters)

## Review the updated map

In [ ]:
# This is also clipped because inHelper._dataset is the 512x512 input clip.
cmDataset = Clusterer.labelsToGeotiff(
    inHelper._dataset,
    clusterMapFile,
    newClusters,
)

cmHelper = ImageHelper()
cmHelper.initFromDataset(cmDataset, noDataValue)

cluster_client = TileClient(str(clusterMapFile), debug=True)
cluster_layer = get_leaflet_tile_layer(
    cluster_client,
    vmin=cmHelper._minValue,
    vmax=cmHelper._maxValue,
    nodata=cmHelper._noDataValue,
    opacity=0.5,
    colormap="viridis",
)
cluster_layer.name = clusterMapFile.name

# Remove the old 30-cluster legend, if it is still on the map
try:
    m.remove(legend_control)
except Exception:
    pass

# Final grouped class IDs
class_ids = sorted(int(x) for x in numpy.unique(newClusters))

vmin = min(class_ids)
vmax = max(class_ids)

cmap = cm.get_cmap("viridis")

rows = []

for class_id in class_ids:
    # Match the same vmin/vmax normalization used by localtileserver
    if vmax == vmin:
        t = 0.5
    else:
        t = (class_id - vmin) / (vmax - vmin)

    hex_color = mcolors.to_hex(cmap(t))

    rows.append(
        f"""
        <div style="
            display:flex;
            align-items:center;
            margin:3px 0;
            color:#111;
        ">
            <div style="
                width:18px;
                height:14px;
                background:{hex_color};
                border:1px solid #444;
                margin-right:8px;
                flex:0 0 auto;
            "></div>

            <div style="
                font-size:12px;
                color:#111;
                white-space:nowrap;
            ">
                Class {class_id}
            </div>
        </div>
        """
    )

final_legend_html = widgets.HTML(
    value=f"""
    <div style="
        background:white;
        color:#111;
        padding:8px 10px;
        border:1px solid #777;
        border-radius:4px;
        min-width:120px;
        box-shadow:0 1px 4px rgba(0,0,0,0.25);
    ">
        <div style="
            font-weight:bold;
            margin-bottom:6px;
            color:#111;
        ">
            Final classes
        </div>

        {''.join(rows)}
    </div>
    """
)

final_legend_control = WidgetControl(
    widget=final_legend_html,
    position="topright",
)

m.add(final_legend_control)

# Remove original first-pass labels
if labels_layer in m.layers:
    m.remove(labels_layer)

# Remove an older final layer if this cell is being rerun
for layer in list(m.layers):
    if layer.name == clusterMapFile.name:
        m.remove(layer)

# Add the newly generated final labels
m.add(cluster_layer)

display(m)